# 12b. Citrus: more seeds of the best model (ConvNeXt-Tiny at 320 px)

In notebook 12 the best Citrus model was **ConvNeXt-Tiny at 320 px with the strong recipe: 96.21 +/- 0.73** (5 folds, one seed), against 95.14 for the plain hybrid (average of two runs), 93.90 for ResNet-50 and 90.61 for the from-scratch Bi-Mamba. The gain over the plain hybrid (+1.08) was positive in 4 of 5 folds but **not statistically clear**.

This notebook runs **two more seeds** of that model on the same 5 folds, so the final claim rests on 3 seeds x 5 folds. It then prints the paired, fold-by-fold gain against each earlier model with its standard error, so you can see whether the gain is clear or only suggestive.

The flip test-time augmentation did not help consistently in notebook 12, so it is ignored here. Resolution did help ConvNeXt a little (224 px 95.72, 320 px 96.21), which is why only the 320 px version is repeated.

Each seed took about 22 minutes in notebook 12, so expect about 50 minutes in total. No smoke test is needed: this code already ran. Results file: `results_12b.csv`.

**Before running:** Accelerator = GPU T4, **Internet ON**, and both plant datasets attached (PlantDoc Classification and Citrus Leaf Disease Image). Safe to run with Save and Run All (Commit): it contains only the runs listed below.

## 1. Install (Internet must be ON)

In [ ]:
!pip install -q mambapy

## 2. Settings

In [ ]:
import os, glob, time, math, random, copy, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from sklearn.metrics import f1_score

# hybrid model (pretrained ResNet-50 tokenizer + Bi-Mamba blocks)
EPOCHS, EPOCHS_STRONG, WARMUP = 30, 45, 2
BATCH = 16                                  # lower to 8 if you get an out-of-memory error
LR, LR_BACKBONE, WD = 5e-4, 1e-4, 0.05
DIM, DEPTH, HYB_DEPTH = 192, 8, 4
# stronger recipe (Mixup / CutMix, RandAugment, weight averaging)
MIX_PROB, MIXUP_ALPHA, CUTMIX_ALPHA, SWITCH, SMOOTH, EMA_DECAY = 0.8, 0.8, 1.0, 0.5, 0.1, 0.998
# plain ResNet-50 reference
RN_EPOCHS, RN_WARMUP, RN_BATCH, RN_LR = 20, 1, 32, 3e-4
RN_EPOCHS_STRONG = 45                      # same number of epochs as the strong hybrid
BB_EPOCHS, BB_BATCH, BB_LR = 45, 32, 2e-4   # stronger pretrained backbones (ConvNeXt-T, Swin-T)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEV == "cuda"
torch.backends.cudnn.benchmark = True
SUFFIX = ""
RESULTS = "/kaggle/working/results_12b.csv"
print("device:", DEV, "| results file:", RESULTS)

## 3. Data (same code and seed as before, so the folds are identical)

In [ ]:
import os, random, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

SEED = 42
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
PD_ROOT = os.environ.get("PD_ROOT", "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset")
CT_ROOT = os.environ.get("CT_ROOT", "/kaggle/input/datasets/myprojectdictionary/citrus-leaf-disease-image/Citrus Leaf Disease Image")
OUT = os.environ.get("SPLIT_OUT", "/kaggle/working/splits")
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
def list_images(root):
    rows = []
    for c in sorted(os.listdir(root)):
        p = os.path.join(root, c)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            if f.lower().endswith(IMG_EXT):
                rows.append((os.path.join(p, f), c))
    return pd.DataFrame(rows, columns=["path", "label"])

def safe_split(df, test_size, seed):
    """Stratified split. Classes with fewer than 2 images cannot be stratified, so they stay in the first part."""
    counts = df.label.value_counts()
    rare = counts[counts < 2].index
    df_rare, df_ok = df[df.label.isin(rare)], df[~df.label.isin(rare)]
    a, b = train_test_split(df_ok, test_size=test_size, stratify=df_ok.label, random_state=seed)
    return pd.concat([a, df_rare]).reset_index(drop=True), b.reset_index(drop=True)

_hash_cache = {}
def file_md5(path):
    if path not in _hash_cache:
        with open(path, "rb") as f:
            _hash_cache[path] = hashlib.md5(f.read()).hexdigest()
    return _hash_cache[path]

def drop_leaks(train, val, test):
    """Remove images from train/val that are byte-identical to a test image (and from train if identical to a val image).
    The test set itself is never changed."""
    test_h = {file_md5(p) for p in test.path}
    val = val[~val.path.map(file_md5).isin(test_h)]
    val_h = {file_md5(p) for p in val.path}
    train = train[~train.path.map(file_md5).isin(test_h | val_h)]
    return train.reset_index(drop=True), val.reset_index(drop=True)

# ---------- PlantDoc: use the provided test set; split train 80/20 into train/val ----------
pd_train_full = list_images(f"{PD_ROOT}/train")
pd_test = list_images(f"{PD_ROOT}/test").reset_index(drop=True)
pd_train, pd_val = safe_split(pd_train_full, 0.20, SEED)
n0 = (len(pd_train), len(pd_val))
pd_train, pd_val = drop_leaks(pd_train, pd_val, pd_test)
print(f"PlantDoc: removed {n0[0]-len(pd_train)} train and {n0[1]-len(pd_val)} val images that duplicate a val/test image")
pd_classes = sorted(pd_train_full.label.unique())          # 28 classes (from train)
pd_map = {c: i for i, c in enumerate(pd_classes)}
unknown = set(pd_test.label) - set(pd_map)
assert not unknown, f"test classes missing from train: {unknown}"

pd_train.to_csv(f"{OUT}/plantdoc_train.csv", index=False)
pd_val.to_csv(f"{OUT}/plantdoc_val.csv", index=False)
pd_test.to_csv(f"{OUT}/plantdoc_test.csv", index=False)
print(f"PlantDoc: train {len(pd_train)} | val {len(pd_val)} | test {len(pd_test)} | classes {len(pd_classes)}")

# ---------- Citrus: no provided split, so 5-fold stratified CV (test = 1 fold, val = 12.5% of the rest) ----------
ct = list_images(CT_ROOT)
ct_classes = sorted(ct.label.unique())
ct_map = {c: i for i, c in enumerate(ct_classes)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ct_folds = []
for k, (tr_idx, te_idx) in enumerate(skf.split(ct, ct.label)):
    rest, test = ct.iloc[tr_idx], ct.iloc[te_idx].reset_index(drop=True)
    train, val = safe_split(rest, 0.125, SEED)
    train, val = drop_leaks(train, val, test)
    ct_folds.append((train, val, test))
    train.to_csv(f"{OUT}/citrus_fold{k}_train.csv", index=False)
    val.to_csv(f"{OUT}/citrus_fold{k}_val.csv", index=False)
    test.to_csv(f"{OUT}/citrus_fold{k}_test.csv", index=False)
    assert not (set(train.path) & set(test.path)) and not (set(val.path) & set(test.path)) and not (set(train.path) & set(val.path))
print(f"Citrus: {len(ct)} images, classes {ct_classes}")
for k, (tr, va, te) in enumerate(ct_folds):
    print(f"  fold {k}: train {len(tr)} | val {len(va)} | test {len(te)}  | melanose in test: {(te.label == 'Melanose').sum()}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, df, cmap, tf):
        self.paths = df.path.tolist()
        self.y = [cmap[l] for l in df.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.y[i]

def make_loaders(train_df, val_df, test_df, cmap, bs=32, workers=2, balanced=False):
    tr = LeafDS(train_df, cmap, train_tf)
    sampler = None
    if balanced:   # class-balanced sampling (the paper reports this setting in Table 11)
        cnt = np.bincount(tr.y, minlength=len(cmap)).astype(float)
        w = [1.0 / max(cnt[y], 1) for y in tr.y]
        sampler = WeightedRandomSampler(w, num_samples=len(tr), replacement=True)
    kw = dict(num_workers=workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, bs, shuffle=sampler is None, sampler=sampler, drop_last=True, **kw),
            DataLoader(LeafDS(val_df, cmap, eval_tf), bs, shuffle=False, **kw),
            DataLoader(LeafDS(test_df, cmap, eval_tf), bs, shuffle=False, **kw))

## 4. Bi-Mamba blocks

In [ ]:
from mambapy.mamba import MambaBlock, MambaConfig

def make_block(d):
    return MambaBlock(MambaConfig(d_model=d, n_layers=1, d_state=16, expand_factor=2,
                                  d_conv=4, pscan=True, use_cuda=False))

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd, self.bwd = make_block(d), make_block(d)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        return x + self.wf * self.fwd(h) + self.wb * self.bwd(h.flip(1)).flip(1)

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=8, patch=16, base_grid=14):
        super().__init__()
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

class Classifier(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=DEPTH):
        super().__init__()
        self.enc = MiniBiMamba(d, depth)
        self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        tokens, _ = self.enc(x)
        return self.head(tokens.mean(1))

_m = Classifier(28)
print("Parameters (M):", round(sum(p.numel() for p in _m.parameters()) / 1e6, 2)); del _m

## 5. Evaluation helpers

In [ ]:
def evaluate(model, loader, n_classes):
    model.eval(); ys, ps, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item()
            ps.append(out.argmax(1).cpu()); ys.append(y.cpu()); n += len(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    f1s = f1_score(y, p, average=None, labels=list(range(n_classes)), zero_division=0)
    present = np.unique(y)                      # PlantDoc test has one class fewer than train
    return dict(loss=loss_sum / n, acc=float((y == p).mean() * 100),
                macro_f1=float(f1s[present].mean() * 100), f1s=f1s, present=present)

def minority_f1(res, train_df, cmap):
    """Mean F1 of the 25% rarest classes (by training count) that appear in the test set."""
    cnt = train_df.label.map(cmap).value_counts()
    thr = cnt.quantile(0.25)
    minority = [c for c in res["present"] if cnt.get(c, 0) <= thr]
    return float(res["f1s"][minority].mean() * 100) if minority else float("nan")

def get_split(name, fold):
    if name == "plantdoc":
        return pd_train, pd_val, pd_test, pd_map
    tr, va, te = ct_folds[fold]
    return tr, va, te, ct_map

def already_done(tag):
    return os.path.exists(RESULTS) and tag in set(pd.read_csv(RESULTS)["tag"])

def run_experiment(name, fold=0, seed=0):
    tag = f"{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = Classifier(n_cls).to(DEV)

    decay = [p for n, p in model.named_parameters() if p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n]
    no_decay = [p for n, p in model.named_parameters() if not (p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WD}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    total, warm = EPOCHS * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ck = f"/kaggle/working/ckpt_baseline{SUFFIX}_{tag}.pt"
    start, best, hist = 0, dict(val_acc=-1.0, epoch=-1, state=None), []
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        scaler.load_state_dict(s["scaler"]); start, best, hist = s["epoch"], s["best"], s["hist"]
        print(f"resumed {tag} from epoch {start}")

    t_start = time.time()
    for ep in range(start, EPOCHS):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue                 # skip a bad step instead of crashing
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1
        r = evaluate(model, lva, n_cls)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), val_loss=r["loss"], val_acc=r["acc"]))
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep + 1, best=best, hist=hist), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
        print(f"[{tag}] epoch {ep+1}/{EPOCHS} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")

    model.load_state_dict(best["state"])                 # test ONLY with the best-validation weights
    res = evaluate(model, lte, n_cls)
    row = dict(tag=tag, dataset=name, fold=fold, seed=seed, epochs=EPOCHS, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
    if os.path.exists(ck): os.remove(ck)                # finished: free the space

## 6. Models and the stronger recipe

In [ ]:
class ResNetTokenizer(nn.Module):
    """ImageNet-pretrained ResNet-50 up to layer3 (stride 16), then a 1x1 projection to the Mamba width.
    It replaces the 16x16 patch-embedding conv: a 224 px image still gives a 14x14 token grid,
    160 px gives 10x10 and 96 px gives 6x6, exactly like the PSMamba views."""
    def __init__(self, d):
        super().__init__()
        r = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)   # needs Internet ON
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3)
        self.proj = nn.Conv2d(1024, d, 1)
    def forward(self, x):
        return self.proj(self.stem(x))                                  # (B, d, gh, gw)

class HybridMamba(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=HYB_DEPTH, base_grid=14):
        super().__init__()
        self.tok = ResNetTokenizer(d)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])    # same Bi-Mamba blocks as before
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)
    def forward(self, img):
        t = self.tok(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.head(self.norm(t).mean(1))

def run_hybrid(name, fold=0, seed=0, label="hybrid", epochs=None):
    ep_total = epochs or EPOCHS
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = HybridMamba(n_cls).to(DEV)
    stem = list(model.tok.stem.parameters()); stem_ids = {id(p) for p in stem}
    rest = [p for p in model.parameters() if id(p) not in stem_ids]
    opt = torch.optim.AdamW([{"params": stem, "lr": LR_BACKBONE, "weight_decay": WD},
                             {"params": rest, "lr": LR, "weight_decay": WD}])
    total, warm = ep_total * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, t_start = dict(val_acc=-1.0, epoch=-1, state=None), time.time()
    for ep in range(ep_total):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1
        r = evaluate(model, lva, n_cls)
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        print(f"[{tag}] epoch {ep+1}/{ep_total} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")
    model.load_state_dict(best["state"])
    res = evaluate(model, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=ep_total, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

In [ ]:
train_tf_strong = T.Compose([
    T.RandomResizedCrop(224, scale=(0.3, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25)])

def mix_batch(x, y, n_cls):
    """Mixup or CutMix with label smoothing. Returns mixed images and soft targets."""
    onehot = F.one_hot(y, n_cls).float() * (1 - SMOOTH) + SMOOTH / n_cls
    if random.random() > MIX_PROB:
        return x, onehot
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < SWITCH:                                   # CutMix
        lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
        H, W = x.shape[-2:]
        rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(cy - rh // 2, 0), min(cy + rh // 2, H)
        x1, x2 = max(cx - rw // 2, 0), min(cx + rw // 2, W)
        x = x.clone()
        x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    else:                                                          # Mixup
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
        x = lam * x + (1 - lam) * x[perm]
    return x, lam * onehot + (1 - lam) * onehot[perm]

def ema_update(ema, model, d):
    """Average weights AND copy the BatchNorm counters (the ResNet part has BatchNorm)."""
    with torch.no_grad():
        for (_, ve), (_, v) in zip(ema.state_dict().items(), model.state_dict().items()):
            if v.dtype.is_floating_point:
                ve.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                ve.copy_(v)

def run_hybrid_strong(name, fold=0, seed=0, label="hybrid_strong", epochs=None):
    global train_tf
    ep_total = epochs or EPOCHS_STRONG
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    train_tf = train_tf_strong                      # make_loaders reads this global
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = HybridMamba(n_cls).to(DEV)
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.requires_grad = False
    stem = list(model.tok.stem.parameters()); stem_ids = {id(p) for p in stem}
    rest = [p for p in model.parameters() if id(p) not in stem_ids]
    opt = torch.optim.AdamW([{"params": stem, "lr": LR_BACKBONE, "weight_decay": WD},
                             {"params": rest, "lr": LR, "weight_decay": WD}])
    total, warm = ep_total * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, step, t_start = dict(val_acc=-1.0, epoch=-1, state=None), 0, time.time()
    for ep in range(ep_total):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            x, soft = mix_batch(x, y, n_cls)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = -(soft * F.log_softmax(out.float(), dim=-1)).sum(-1).mean()
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema_update(ema, model, min(EMA_DECAY, (1 + step) / (10 + step)))
            step += 1; tl += loss.item(); nb += 1
        r = evaluate(ema, lva, n_cls)                                   # validate the averaged weights
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in ema.state_dict().items()})
        print(f"[{tag}] epoch {ep+1}/{ep_total} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")
    ema.load_state_dict(best["state"])
    res = evaluate(ema, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=ep_total, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

In [ ]:
def make_backbone(arch, n_cls):
    if arch == "convnext_tiny":
        m = torchvision.models.convnext_tiny(weights=torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)   # needs Internet ON
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_cls)
    elif arch == "swin_t":
        m = torchvision.models.swin_t(weights=torchvision.models.Swin_T_Weights.IMAGENET1K_V1)
        m.head = nn.Linear(m.head.in_features, n_cls)
    else:
        raise ValueError(arch)
    return m

def run_backbone_strong(name, fold=0, seed=0, arch="convnext_tiny", epochs=None):
    global train_tf
    ep_total = epochs or BB_EPOCHS
    label = f"{arch}_strong"
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    train_tf = train_tf_strong
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BB_BATCH)
    n_cls = len(cmap)
    model = make_backbone(arch, n_cls).to(DEV)
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.requires_grad = False
    opt = torch.optim.AdamW(model.parameters(), lr=BB_LR, weight_decay=WD)
    total, warm = ep_total * len(ltr), 2 * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, step, t_start = dict(val_acc=-1.0, epoch=-1, state=None), 0, time.time()
    for ep in range(ep_total):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            x, soft = mix_batch(x, y, n_cls)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = -(soft * F.log_softmax(out.float(), dim=-1)).sum(-1).mean()
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema_update(ema, model, min(EMA_DECAY, (1 + step) / (10 + step)))
            step += 1; tl += loss.item(); nb += 1
        r = evaluate(ema, lva, n_cls)
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in ema.state_dict().items()})
        print(f"[{tag}] epoch {ep+1}/{ep_total} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")
    ema.load_state_dict(best["state"])
    res = evaluate(ema, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=ep_total, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

## 7. Citrus runner

In [ ]:
from sklearn.metrics import confusion_matrix

EVAL_TF_224 = eval_tf

def strong_tf(size):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.3, 1.0)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandAugment(num_ops=2, magnitude=9),
        T.ColorJitter(0.2, 0.2, 0.2, 0.02),
        T.ToTensor(), T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25)])

@torch.no_grad()
def predict(model, loader, tta):
    """Class probabilities. With tta=True the prediction averages the image, its horizontal flip and its vertical flip."""
    model.eval(); P, Y = [], []
    for x, y in loader:
        x = x.to(DEV, non_blocking=True)
        views = [x, x.flip(3), x.flip(2)] if tta else [x]
        prob = 0
        for v in views:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                prob = prob + F.softmax(model(v).float(), dim=-1)
        P.append((prob / len(views)).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y)

def score(P, Y, n_cls):
    p, y = P.argmax(1).numpy(), Y.numpy()
    f1s = f1_score(y, p, average=None, labels=list(range(n_cls)), zero_division=0)
    return float((p == y).mean() * 100), float(f1s[np.unique(y)].mean() * 100), confusion_matrix(y, p, labels=list(range(n_cls)))

ARMS = {
    "hybrid_strong": dict(kind="hybrid",        img=224, epochs=45, bs=16, lr=5e-4),
    "convnext_224":  dict(kind="convnext_tiny", img=224, epochs=45, bs=32, lr=2e-4),
    "convnext_320":  dict(kind="convnext_tiny", img=320, epochs=45, bs=32, lr=2e-4),
}

def run_arm(arm, fold, seed=0, epochs=None):
    global train_tf, eval_tf
    cfg = ARMS[arm]
    smoke = epochs is not None
    ep_total = epochs or cfg["epochs"]
    label = "smoke" if smoke else arm
    tag = f"{'smoke_' if smoke else ''}{arm}_citrus_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    train_tf = strong_tf(cfg["img"])
    eval_tf = T.Compose([T.Resize((cfg["img"], cfg["img"])), T.ToTensor(), T.Normalize(MEAN, STD)])
    try:
        tr, va, te, cmap = get_split("citrus", fold)
        ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=cfg["bs"])
        n_cls = len(cmap)
        model = (HybridMamba(n_cls) if cfg["kind"] == "hybrid" else make_backbone(cfg["kind"], n_cls)).to(DEV)
        ema = copy.deepcopy(model)
        for p in ema.parameters():
            p.requires_grad = False
        if cfg["kind"] == "hybrid":
            stem = list(model.tok.stem.parameters()); ids = {id(p) for p in stem}
            rest = [p for p in model.parameters() if id(p) not in ids]
            opt = torch.optim.AdamW([{"params": stem, "lr": LR_BACKBONE, "weight_decay": WD},
                                     {"params": rest, "lr": cfg["lr"], "weight_decay": WD}])
        else:
            opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=WD)
        total, warm = ep_total * len(ltr), 2 * len(ltr)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        best, step, t_start = dict(val_acc=-1.0, epoch=-1, state=None), 0, time.time()
        for ep in range(ep_total):
            model.train(); t0, tl, nb = time.time(), 0.0, 0
            for x, y in ltr:
                x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
                x, soft = mix_batch(x, y, n_cls)
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    out = model(x)
                loss = -(soft * F.log_softmax(out.float(), dim=-1)).sum(-1).mean()
                opt.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):
                    sched.step(); continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                ema_update(ema, model, min(EMA_DECAY, (1 + step) / (10 + step)))
                step += 1; tl += loss.item(); nb += 1
            r = evaluate(ema, lva, n_cls)
            if r["acc"] > best["val_acc"]:
                best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in ema.state_dict().items()})
            if (ep + 1) % 5 == 0 or ep == 0 or ep + 1 == ep_total:
                print(f"[{tag}] epoch {ep+1}/{ep_total} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")
        ema.load_state_dict(best["state"])
        P, Y = predict(ema, lte, tta=False); acc, mf1, cm = score(P, Y, n_cls)
        P2, _ = predict(ema, lte, tta=True); acc_t, mf1_t, _ = score(P2, Y, n_cls)
        row = dict(tag=tag, label=label, arm=arm, dataset="citrus", fold=fold, seed=seed, epochs=ep_total,
                   best_epoch=best["epoch"] + 1, val_acc=best["val_acc"], test_acc=acc, test_macro_f1=mf1,
                   test_acc_tta=acc_t, test_macro_f1_tta=mf1_t, minutes=(time.time() - t_start) / 60)
        pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
        print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
        print("confusion matrix (rows = true class, columns = predicted), classes:", sorted(cmap, key=cmap.get))
        print(cm)
    finally:
        eval_tf = EVAL_TF_224

## 8. Runs: ConvNeXt-Tiny at 320 px, seeds 1 and 2, all 5 folds each

In [ ]:
for sd in (1, 2):
    for k in range(5):
        run_arm("convnext_320", fold=k, seed=sd)

## Result

In [ ]:
# per-fold test accuracy of the runs you already have (Citrus, folds 0 to 4)
PRIOR = {
    "scratch Bi-Mamba":          [91.80, 90.98, 90.91, 92.56, 86.78],
    "ResNet-50 plain":           [95.08, 95.08, 92.56, 94.21, 92.56],
    "hybrid plain, run 1":       [95.90, 96.72, 93.39, 95.04, 95.87],
    "hybrid plain, run 2":       [95.08, 96.72, 91.74, 94.21, 96.69],
    "ConvNeXt-T 320, seed 0":    [95.90, 95.90, 95.87, 95.87, 97.52],
}
df = pd.read_csv(RESULTS)
df = df[df.label != "smoke"]
runs = {"ConvNeXt-T 320, seed 0": np.array(PRIOR["ConvNeXt-T 320, seed 0"])}
for sd in sorted(df.seed.unique()):
    d = df[df.seed == sd].sort_values("fold")
    if len(d) == 5:
        runs[f"ConvNeXt-T 320, seed {sd}"] = d.test_acc.values
    else:
        print(f"seed {sd}: only {len(d)} of 5 folds finished, skipped")
hyb = np.mean([PRIOR["hybrid plain, run 1"], PRIOR["hybrid plain, run 2"]], axis=0)
print("Per-fold test accuracy (folds 0 to 4):")
for k, v in PRIOR.items():
    if k.startswith("ConvNeXt"): continue
    print(f"  {k:26s} {np.round(v, 1)}  mean {np.mean(v):6.2f} +/- {np.std(v, ddof=1):4.2f}")
for k, v in runs.items():
    print(f"  {k:26s} {np.round(v, 1)}  mean {np.mean(v):6.2f} +/- {np.std(v, ddof=1):4.2f}")
cn = np.mean(list(runs.values()), axis=0)      # per-fold average over the ConvNeXt seeds
print()
print(f"ConvNeXt-T 320 averaged over {len(runs)} seeds, per fold: {np.round(cn, 1)}  mean {cn.mean():.2f}")
for name, ref in (("plain hybrid (avg of 2 runs)", hyb), ("ResNet-50 plain", np.array(PRIOR["ResNet-50 plain"])),
                  ("scratch Bi-Mamba", np.array(PRIOR["scratch Bi-Mamba"]))):
    g = cn - ref
    se = g.std(ddof=1) / np.sqrt(len(g))
    print(f"  gain vs {name:30s}: {g.mean():+.2f}  (per fold {np.round(g, 1)}), standard error {se:.2f}, ratio {g.mean()/se:.1f}")
print()
print("A ratio above about 2.8 (4 degrees of freedom) is needed to call the gain clear; below that, say 'suggestive'.")